# 04 — Feature Engineering & Model Preparation

## Project

**MaternalRisk: Early Prediction of Gestational Diabetes Using Explainable Machine Learning**

## Objective

Prepare two reproducible modeling scenarios for gestational diabetes prediction:

1. **Early Pregnancy Model** — uses information reasonably available at or near the first prenatal visit.
2. **Post-OGTT Comparison Model** — uses the same predictors plus the Oral Glucose Tolerance Test result.

This notebook will:

- Define and document both feature sets.
- Separate predictors from the target.
- Create one shared stratified train/test split.
- Develop leakage-safe preprocessing pipelines.
- Define missing-value and scaling strategies.
- Save the artifacts required for baseline modeling.

## Background

Exploratory analysis identified meaningful relationships between gestational diabetes and several maternal demographic, medical-history, and clinical variables.

The primary research objective is early risk assessment. Therefore, the primary model must use only information that could plausibly be available before routine gestational diabetes screening. A secondary comparison model will include OGTT to quantify how predictive performance changes once glucose-testing information becomes available.

All preprocessing steps that learn from the data—including imputation and scaling—will be fitted using the training set only. This prevents information from the test set from leaking into model development.

## Workflow

1. Load the processed dataset.
2. Define the target and two modeling feature sets.
3. Review feature roles and prediction-time availability.
4. Create a shared stratified train/test split.
5. Define missing-value strategies.
6. Build reproducible preprocessing pipelines.
7. Validate transformed outputs.
8. Save the split indices and preprocessing artifacts for modeling.

In [100]:
from pathlib import Path

import joblib
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [101]:
DATA_PATH = Path("../data/processed/gdm_clean.parquet")
ARTIFACTS_DIR = Path("../models/preprocessing")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Class Label(GDM /Non GDM)"
RANDOM_STATE = 42
TEST_SIZE = 0.20

In [102]:
df = pd.read_parquet(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Target values: {sorted(df[TARGET].unique())}")

df.head()

Dataset shape: (3525, 17)
Target values: [np.int64(0), np.int64(1)]


,Age,No of Pregnancy,Gestation in previous Pregnancy,BMI,HDL,Family History,unexplained prenetal loss,Large Child or Birth Default,PCOS,Sys BP,Dia BP,OGTT,Hemoglobin,Sedentary Lifestyle,Prediabetes,Class Label(GDM /Non GDM),Missing Count
0,22,2,1,NaN,55.0,0,0,0,0,102.0,69,NaN,12.0,0,0,0,2
1,26,2,1,NaN,53.0,0,0,0,0,101.0,63,NaN,12.4,0,0,0,2
2,29,1,0,NaN,50.0,0,0,0,0,118.0,79,NaN,14.3,0,0,0,2
3,28,2,1,NaN,51.0,0,0,0,0,99.0,70,NaN,15.0,0,0,0,2
4,21,2,1,NaN,52.0,0,0,0,0,116.0,65,NaN,15.0,0,0,0,2


In [103]:
if TARGET not in df.columns:
    raise KeyError(f"Target column not found: {TARGET}")

if not set(df[TARGET].dropna().unique()).issubset({0, 1}):
    raise ValueError("Target must be binary and encoded as 0/1.")

## Investigation 1 — Feature Selection and Prediction Timing

## Question

Which predictors belong in the early-pregnancy model, and which additional information should be reserved for the post-OGTT comparison model?

In [104]:
early_features = [
    "Age",
    "No of Pregnancy",
    "Gestation in previous Pregnancy",
    "BMI",
    "HDL",
    "Family History",
    "unexplained prenetal loss",
    "Large Child or Birth Default",
    "PCOS",
    "Sys BP",
    "Dia BP",
    "Hemoglobin",
    "Sedentary Lifestyle",
    "Prediabetes",
]

diagnostic_features = early_features + ["OGTT"]

In [105]:
assert len(early_features) == len(set(early_features))
assert len(diagnostic_features) == len(set(diagnostic_features))

In [106]:
required_columns = set(diagnostic_features + [TARGET])
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise KeyError(f"Expected columns are missing: {sorted(missing_columns)}")

In [107]:
feature_design = pd.DataFrame(
    [
        ["Age", "Early", "Known at intake"],
        ["No of Pregnancy", "Early", "Obstetric history"],
        ["Gestation in previous Pregnancy", "Early — verify", "Prior pregnancy history; definition requires confirmation"],
        ["BMI", "Early", "Height and weight can be collected at first visit"],
        ["HDL", "Early — verify", "May require laboratory testing not universally available at intake"],
        ["Family History", "Early", "Patient history"],
        ["unexplained prenetal loss", "Early", "Obstetric history"],
        ["Large Child or Birth Default", "Early", "Prior obstetric history"],
        ["PCOS", "Early", "Pre-existing history"],
        ["Sys BP", "Early", "Measured during prenatal care"],
        ["Dia BP", "Early", "Measured during prenatal care"],
        ["Hemoglobin", "Early — verify", "Often measured early but timing may vary"],
        ["Sedentary Lifestyle", "Early", "Patient-reported characteristic"],
        ["Prediabetes", "Early", "Pre-existing history"],
        ["OGTT", "Post-screening", "Typically associated with later GDM screening"],
    ],
    columns=["Feature", "Modeling Role", "Rationale"],
)

feature_design

,Feature,Modeling Role,Rationale
0,Age,Early,Known at intake
1,No of Pregnancy,Early,Obstetric history
2,Gestation in previous Pregnancy,Early — verify,Prior pregnancy history; definition requires c...
3,BMI,Early,Height and weight can be collected at first visit
4,HDL,Early — verify,May require laboratory testing not universally...
5,Family History,Early,Patient history
6,unexplained prenetal loss,Early,Obstetric history
7,Large Child or Birth Default,Early,Prior obstetric history
8,PCOS,Early,Pre-existing history
9,Sys BP,Early,Measured during prenatal care


The feature inventory confirms that the selected predictors naturally separate into two clinically meaningful modeling scenarios: variables available before routine gestational diabetes screening and variables available after diagnostic glucose testing.

## Interpretation

Two nested feature sets will be used. The primary feature set excludes `OGTT` and is intended to approximate information available before routine gestational diabetes screening. The comparison feature set contains the same predictors plus `OGTT`.

Several measurements—particularly `HDL` and `Hemoglobin`—are provisionally treated as early-pregnancy features because they may be available during early prenatal care, but their exact timing in the source dataset remains uncertain. This assumption will be documented as a limitation and revisited if the original study provides clearer measurement timing.

## Decision Point 1 — Two Modeling Scenarios

**Decision**

Develop an Early Pregnancy Model without `OGTT` and a Post-OGTT Comparison Model that includes it.

**Rationale**

The primary research question concerns risk estimation before routine glucose screening. Including `OGTT` in the primary model would weaken that interpretation because OGTT is closely connected to GDM diagnosis. The comparison model will quantify the additional predictive information available after glucose testing.

**Consequence**

Both scenarios must use the same patient split and evaluation procedure so that differences in performance can be attributed to feature availability rather than different samples.

## Investigation 2 — Train/Test Partition

## Question

How should the data be partitioned to support fair and reproducible comparison between the two modeling scenarios?

In [108]:
train_index, test_index = train_test_split(
    df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[TARGET],
)

train_df = df.loc[train_index].copy()
test_df = df.loc[test_index].copy()

y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

X_train_early = train_df[early_features].copy()
X_test_early = test_df[early_features].copy()

X_train_diagnostic = train_df[diagnostic_features].copy()
X_test_diagnostic = test_df[diagnostic_features].copy()

In [109]:
split_summary = pd.DataFrame(
    {
        "Full Dataset (%)": df[TARGET].value_counts(normalize=True).sort_index() * 100,
        "Training Set (%)": y_train.value_counts(normalize=True).sort_index() * 100,
        "Test Set (%)": y_test.value_counts(normalize=True).sort_index() * 100,
    }
).round(2)

split_summary.index = ["Non-GDM", "GDM"]
split_summary

,Full Dataset (%),Training Set (%),Test Set (%)
Non-GDM,61.08,61.06,61.13
GDM,38.92,38.94,38.87


In [110]:
assert set(train_index).isdisjoint(set(test_index))
assert len(train_index) + len(test_index) == len(df)

print(f"Training rows: {len(train_index)}")
print(f"Test rows: {len(test_index)}")

Training rows: 2820
Test rows: 705


## Decision Point 2 — Shared Stratified Split

**Decision**

Use a single 80/20 stratified train/test split for both modeling scenarios.

**Rationale**

Stratification preserves the GDM class distribution in both subsets. Reusing the same patient split ensures a fair comparison between the Early Pregnancy and Post-OGTT models.

**Consequence**

- Both models will use identical training and test observations.
- The test set will remain untouched during preprocessing decisions, model selection, hyperparameter tuning, and threshold optimization.

## Investigation 3 — Missing-Value Strategy

## Question

How should missing predictor values be handled while preserving potentially informative missingness patterns and preventing data leakage?

In [111]:
missing_train = pd.DataFrame({
    "Missing Count": X_train_diagnostic.isna().sum(),
    "Missing (%)": (
        X_train_diagnostic.isna().mean() * 100
    ).round(2)
})

missing_train = missing_train[missing_train["Missing Count"] > 0]

missing_train

,Missing Count,Missing (%)
BMI,881,31.24
HDL,810,28.72
Sys BP,1364,48.37
OGTT,422,14.96


In [112]:
binary_features = [
    "Family History",
    "unexplained prenetal loss",
    "Large Child or Birth Default",
    "PCOS",
    "Sedentary Lifestyle",
    "Prediabetes",
]

In [113]:
continuous_early_features = [
    "Age",
    "No of Pregnancy",
    "Gestation in previous Pregnancy",
    "BMI",
    "HDL",
    "Sys BP",
    "Dia BP",
    "Hemoglobin",
]

continuous_diagnostic_features = continuous_early_features + ["OGTT"]

In [114]:
assert set(continuous_early_features + binary_features) == set(early_features)

assert set(continuous_diagnostic_features + binary_features) == set(
    diagnostic_features
)

In [115]:
SimpleImputer(
    strategy="median",
    add_indicator=True
)

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",True
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False


In [116]:
X_train_diagnostic[binary_features].isna().sum()

Family History                  0
unexplained prenetal loss       0
Large Child or Birth Default    0
PCOS                            0
Sedentary Lifestyle             0
Prediabetes                     0
dtype: int64

In [117]:
X_train_diagnostic[continuous_diagnostic_features].isna().sum()

Age                                   0
No of Pregnancy                       0
Gestation in previous Pregnancy       0
BMI                                 881
HDL                                 810
Sys BP                             1364
Dia BP                                0
Hemoglobin                            0
OGTT                                422
dtype: int64

## Interpretation

Missingness in the training data is limited to four continuous clinical measurements: BMI, HDL, systolic blood pressure, and OGTT. Systolic blood pressure has the highest proportion of missing observations at 48.37%, followed by BMI (31.24%), HDL (28.72%), and OGTT (14.96%). All binary predictors and the remaining continuous predictors are complete.

Given the substantial missingness in several measurements, complete-case analysis would result in considerable data loss and is therefore not appropriate. Median imputation will be used for continuous variables with missing values because it is relatively robust to the skewness and clinically plausible extreme observations identified during exploratory analysis.

Previous analysis also demonstrated that measurement availability is associated with gestational diabetes status. Missingness indicators will therefore be retained so that models can distinguish observed measurements from values introduced through imputation.

All imputation parameters will be learned exclusively from the training data through the preprocessing pipeline, preventing information from the test set from influencing model development.

## Decision Point 3 — Missing-Value Handling

**Decision**

Apply median imputation to continuous predictors with missing values and retain missingness indicators for those variables. Binary predictors and complete continuous predictors do not require imputation.

**Rationale**

Four continuous measurements contain missing data, including nearly half of systolic blood pressure observations. Removing incomplete records would substantially reduce the available training data and could introduce additional selection bias. Median imputation provides a robust replacement strategy, while missingness indicators preserve potentially informative patterns in measurement availability.

**Consequence**

Imputation and missingness-indicator creation will be implemented within the Scikit-Learn preprocessing pipeline and fitted using training data only. The test set will be transformed using parameters learned exclusively from the training set.

## Investigation 4 — Preprocessing and Feature Scaling

## Question

Which preprocessing transformations should be applied to each feature type to support reproducible model development?

In [118]:
continuous_missing_early = [
    "BMI",
    "HDL",
    "Sys BP",
]

continuous_missing_diagnostic = [
    "BMI",
    "HDL",
    "Sys BP",
    "OGTT",
]

continuous_complete = [
    "Age",
    "No of Pregnancy",
    "Gestation in previous Pregnancy",
    "Dia BP",
    "Hemoglobin",
]

In [119]:
assert set(
    continuous_missing_early
    + continuous_complete
    + binary_features
) == set(early_features)

assert set(
    continuous_missing_diagnostic
    + continuous_complete
    + binary_features
) == set(diagnostic_features)

In [120]:
# Define the continuous missing pipeline
linear_missing_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        ("scaler", StandardScaler()),
    ]
)

linear_complete_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
    ]
)

tree_missing_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ]
)

In [121]:
# Build early-pregnancy preprocessor
early_linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous_missing",
            linear_missing_pipeline,
            continuous_missing_early,
        ),
        (
            "continuous_complete",
            linear_complete_pipeline,
            continuous_complete,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

early_tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous_missing",
            tree_missing_pipeline,
            continuous_missing_early,
        ),
        (
            "continuous_complete",
            "passthrough",
            continuous_complete,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [122]:
# Build post-OGTT preprocessor
diagnostic_linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous_missing",
            linear_missing_pipeline,
            continuous_missing_diagnostic,
        ),
        (
            "continuous_complete",
            linear_complete_pipeline,
            continuous_complete,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

diagnostic_tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous_missing",
            tree_missing_pipeline,
            continuous_missing_diagnostic,
        ),
        (
            "continuous_complete",
            "passthrough",
            continuous_complete,
        ),
        (
            "binary",
            "passthrough",
            binary_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [123]:
X_train_early_linear = early_linear_preprocessor.fit_transform(
    X_train_early
)

X_train_early_tree = early_tree_preprocessor.fit_transform(
    X_train_early
)

X_train_diagnostic_linear = diagnostic_linear_preprocessor.fit_transform(
    X_train_diagnostic
)

X_train_diagnostic_tree = diagnostic_tree_preprocessor.fit_transform(
    X_train_diagnostic
)

In [124]:
early_linear_feature_names = (
    early_linear_preprocessor.get_feature_names_out()
)

diagnostic_linear_feature_names = (
    diagnostic_linear_preprocessor.get_feature_names_out()
)

print("Early linear features:")
print(early_linear_feature_names)

print("\nDiagnostic linear features:")
print(diagnostic_linear_feature_names)

Early linear features:
['BMI' 'HDL' 'Sys BP' 'missingindicator_BMI' 'missingindicator_HDL'
 'missingindicator_Sys BP' 'Age' 'No of Pregnancy'
 'Gestation in previous Pregnancy' 'Dia BP' 'Hemoglobin' 'Family History'
 'unexplained prenetal loss' 'Large Child or Birth Default' 'PCOS'
 'Sedentary Lifestyle' 'Prediabetes']

Diagnostic linear features:
['BMI' 'HDL' 'Sys BP' 'OGTT' 'missingindicator_BMI' 'missingindicator_HDL'
 'missingindicator_Sys BP' 'missingindicator_OGTT' 'Age' 'No of Pregnancy'
 'Gestation in previous Pregnancy' 'Dia BP' 'Hemoglobin' 'Family History'
 'unexplained prenetal loss' 'Large Child or Birth Default' 'PCOS'
 'Sedentary Lifestyle' 'Prediabetes']


In [125]:
print(
    "Early linear shape:",
    X_train_early_linear.shape,
)

print(
    "Early tree shape:",
    X_train_early_tree.shape,
)

print(
    "Diagnostic linear shape:",
    X_train_diagnostic_linear.shape,
)

print(
    "Diagnostic tree shape:",
    X_train_diagnostic_tree.shape,
)

Early linear shape: (2820, 17)
Early tree shape: (2820, 17)
Diagnostic linear shape: (2820, 19)
Diagnostic tree shape: (2820, 19)


## Interpretation

The preprocessing architecture successfully separates transformations by model family and feature type.

For Logistic Regression, continuous predictors are standardized after median imputation where necessary, while binary predictors remain encoded as 0/1. For Random Forest and XGBoost, continuous predictors are imputed but left unscaled because tree-based models do not require standardized feature magnitudes.

Missingness indicators are created only for continuous variables containing missing observations. This expands the early-pregnancy feature space by three indicators and the post-OGTT feature space by four indicators, allowing the models to retain information about measurement availability.

All preprocessing objects are fitted exclusively on the training data.

## Decision Point 4 — Model-Specific Preprocessing

**Decision**

Use separate preprocessing pipelines for linear and tree-based models.

**Rationale**

Logistic Regression benefits from standardized continuous predictors, particularly when regularization is used. Random Forest and XGBoost rely on feature thresholds and therefore do not require scaling. Both preprocessing variants use median imputation and preserve missingness indicators for incomplete continuous predictors.

**Consequence**

Each model family receives preprocessing appropriate to its mathematical structure while maintaining a consistent and leakage-safe feature engineering strategy.

## Investigation 5 — Preprocessing Validation

## Question

Do the preprocessing pipelines produce complete, correctly structured, and leakage-safe feature matrices for both modeling scenarios?

In [126]:
transformed_sets = {
    "Early Linear": X_train_early_linear,
    "Early Tree": X_train_early_tree,
    "Diagnostic Linear": X_train_diagnostic_linear,
    "Diagnostic Tree": X_train_diagnostic_tree,
}

for name, X_transformed in transformed_sets.items():
    print(
        f"{name}: "
        f"{np.isnan(X_transformed).sum()} missing values"
    )

Early Linear: 0 missing values
Early Tree: 0 missing values
Diagnostic Linear: 0 missing values
Diagnostic Tree: 0 missing values


In [127]:
assert X_train_early_linear.shape[0] == len(X_train_early)
assert X_train_early_tree.shape[0] == len(X_train_early)

assert X_train_diagnostic_linear.shape[0] == len(
    X_train_diagnostic
)

assert X_train_diagnostic_tree.shape[0] == len(
    X_train_diagnostic
)

print("All transformed datasets preserve the training row count.")

All transformed datasets preserve the training row count.


In [128]:
assert X_train_early_linear.shape[1] == 17
assert X_train_early_tree.shape[1] == 17

assert X_train_diagnostic_linear.shape[1] == 19
assert X_train_diagnostic_tree.shape[1] == 19

print("All transformed datasets contain the expected number of features.")

All transformed datasets contain the expected number of features.


In [129]:
X_test_early_linear = early_linear_preprocessor.transform(
    X_test_early
)

X_test_early_tree = early_tree_preprocessor.transform(
    X_test_early
)

X_test_diagnostic_linear = (
    diagnostic_linear_preprocessor.transform(
        X_test_diagnostic
    )
)

X_test_diagnostic_tree = (
    diagnostic_tree_preprocessor.transform(
        X_test_diagnostic
    )
)

In [130]:
test_transformed_sets = {
    "Early Linear": X_test_early_linear,
    "Early Tree": X_test_early_tree,
    "Diagnostic Linear": X_test_diagnostic_linear,
    "Diagnostic Tree": X_test_diagnostic_tree,
}

for name, X_transformed in test_transformed_sets.items():
    print(
        f"{name}: "
        f"{X_transformed.shape[0]} rows, "
        f"{X_transformed.shape[1]} features, "
        f"{np.isnan(X_transformed).sum()} missing values"
    )

Early Linear: 705 rows, 17 features, 0 missing values
Early Tree: 705 rows, 17 features, 0 missing values
Diagnostic Linear: 705 rows, 19 features, 0 missing values
Diagnostic Tree: 705 rows, 19 features, 0 missing values


In [131]:
early_tree_feature_names = (
    early_tree_preprocessor.get_feature_names_out()
)

diagnostic_tree_feature_names = (
    diagnostic_tree_preprocessor.get_feature_names_out()
)

assert np.array_equal(
    early_linear_feature_names,
    early_tree_feature_names,
)

assert np.array_equal(
    diagnostic_linear_feature_names,
    diagnostic_tree_feature_names,
)

print("Feature names are consistent across preprocessing variants.")

Feature names are consistent across preprocessing variants.


## Interpretation

All preprocessing pipelines successfully produce complete modeling matrices with no remaining missing values. The transformations preserve the number of patient observations while expanding the feature space only through the expected missingness indicators.

The Early Pregnancy scenario contains 17 transformed features, while the Post-OGTT scenario contains 19. Feature definitions remain consistent between the linear and tree-based preprocessing variants; the primary difference is the standardization applied to continuous predictors for Logistic Regression.

The fitted preprocessors also successfully transform the held-out test data without refitting, confirming that preprocessing parameters learned from the training set can be applied consistently to unseen observations.

## Decision Point 5 — Preprocessing Validation

**Decision**

Retain the validated model-specific preprocessing architecture for subsequent model development.

**Rationale**

All pipelines preserve patient observations, eliminate missing values through training-derived imputation, generate the expected missingness indicators, and maintain consistent feature definitions across model families. Held-out test data can be transformed without refitting the preprocessing components.

**Consequence**

The preprocessing workflow is ready to be integrated directly with machine-learning estimators using Scikit-Learn Pipelines, ensuring that the same transformations are applied consistently during training, cross-validation, and final evaluation.

In [132]:
joblib.dump(
    early_linear_preprocessor,
    ARTIFACTS_DIR / "early_linear_preprocessor.joblib"
)

joblib.dump(
    early_tree_preprocessor,
    ARTIFACTS_DIR / "early_tree_preprocessor.joblib"
)

joblib.dump(
    diagnostic_linear_preprocessor,
    ARTIFACTS_DIR / "diagnostic_linear_preprocessor.joblib"
)

joblib.dump(
    diagnostic_tree_preprocessor,
    ARTIFACTS_DIR / "diagnostic_tree_preprocessor.joblib"
)

['..\\models\\preprocessing\\diagnostic_tree_preprocessor.joblib']

In [133]:
split_indices = {
    "train_index": train_index.tolist(),
    "test_index": test_index.tolist(),
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
}

joblib.dump(
    split_indices,
    ARTIFACTS_DIR / "train_test_split.joblib"
)

['..\\models\\preprocessing\\train_test_split.joblib']

## Summary

This notebook prepared the gestational diabetes dataset for reproducible machine learning by defining two clinically distinct modeling scenarios and implementing leakage-safe preprocessing workflows.

Key outcomes included:

* Defined an **Early Pregnancy Model** using predictors available before routine glucose screening.
* Defined a **Post-OGTT Comparison Model** that incorporates OGTT measurements.
* Created a single stratified train/test split shared across both modeling scenarios.
* Identified continuous predictors requiring missing-value imputation.
* Selected median imputation with missingness indicators based on the data quality and exploratory analyses.
* Developed model-specific preprocessing pipelines for linear and tree-based algorithms.
* Applied standardization to continuous predictors for Logistic Regression while retaining unscaled values for Random Forest and XGBoost.
* Validated that all preprocessing pipelines preserve patient counts, eliminate missing values, generate the expected feature sets, and transform held-out test data without refitting.
* Saved preprocessing objects and split metadata to support reproducible downstream model development.

The validated preprocessing architecture provides a consistent foundation for comparing model families while minimizing data leakage and preserving the clinical interpretation of the two prediction scenarios.

## Next Steps

The next notebook will establish an interpretable baseline using Logistic Regression.

Both the Early Pregnancy and Post-OGTT feature sets will be evaluated using the same training and test observations. Model performance will be assessed using clinically meaningful metrics including ROC-AUC, recall, precision, F1-score, specificity, and calibration.

Cross-validation will be performed using complete Scikit-Learn pipelines so that imputation and scaling are learned independently within each training fold. This approach will preserve the leakage-safe design established in this notebook and provide a reliable baseline for subsequent comparison with Random Forest and XGBoost.
